# Create Business Finland Awards

Creates Business Finland awards from Finland's national research-information hub **research.fi**.

**Prerequisites:** run `scripts/local/business_finland_to_s3.py` first (queries the research.fi portal Elasticsearch `funding` index, filtered to `funderNameEn = "Innovaatiorahoituskeskus Business Finland"` -> 2,342 funding decisions; writes parquet).

**Data source:** `https://researchfi-api-production.2.rahtiapp.fi/portalapi/funding/_search` (public, no auth). The `funding` index holds 23,023 Finnish funding decisions across all funders; we take the Business Finland subset.

**S3 location:** `s3a://openalex-ingest/awards/business_finland/business_finland_projects.parquet`

**OpenAlex funder:** Business Finland · `funder_id 4320328501` · FI · 5,543 works.
**NOTE — funder identity:** this is a *separate* OpenAlex funder from **Tekes** (`F4320321855`), its pre-2018 predecessor. research.fi carries the Business-Finland-era decisions (diary numbers 2017-2027), so these are correctly attributed to `4320328501`. Pre-2018 Tekes grants would need a different source.

**Schema notes:**
- `funder_award_id` = `funderProjectNumber`, the Business Finland diary number (e.g. `6867/31/2019`) — **the form cited in BF-funded papers** (100% present; this is the value of the ingest).
- `amount` = EUR (98.9% present). 0/neg -> NULL (never fillna 0).
- `lead_investigator` = funding contact person (given=FirstNames, family=LastName); 52% present (company-led co-innovation grants often have no named academic contact). `affiliation.name` = the specific recipient org (`consortiumOrganizationNameEn`, e.g. "Tampere University"), 100% present, country Finland.
- `display_name` = project name (EN preferred, else FI/SV — Finnish-dominant).
- `start_year`/`end_year` = `fundingStartYear`/`fundingEndYear`.

**Provenance** `research_fi_business_finland`, **priority 213** (Claude = odd slots).

**REUSABLE:** research.fi's `funding` index covers many Finnish funders. To onboard another (Svenska kulturfonden `F4320324257`, etc.) rerun the scraper with `--funder-name/--provenance/--slug` and clone this notebook with the new funder_id.

**⚠️ §6.4a NOTE — high-frequency PI combos are REAL, not a scraper bug.** Top contact-person combos with n>5 — e.g. (Tommi, Mikkonen)=12, (Emil, Kurvinen)=11, (Markku, Turunen)=10, (Mikko, Valkama)=10 — are prolific Finnish professors legitimately leading many Business Finland co-innovation projects (verified: diverse project titles across real universities — Jyväskylä, Helsinki, Tampere, Oulu). 667 distinct PIs across 1,217 named rows; 0 institution-as-person. Data is from a clean JSON API (not DOM scraping). Do NOT NULL them.

## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.business_finland_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/business_finland/business_finland_projects.parquet`;

In [ ]:
%sql
SELECT COUNT(*) FROM openalex.awards.business_finland_raw;

In [ ]:
%sql
DESCRIBE openalex.awards.business_finland_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.business_finland_raw LIMIT 5;

## Step 1.6: Funder existence fail-fast
Must return exactly 1 row. If 0, STOP — funder missing from `openalex.common.funder`.

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi
FROM openalex.common.funder WHERE funder_id = 4320328501;

## Step 2: Create Business Finland Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.business_finland_awards
USING delta
AS
WITH
bf_funder AS (
    SELECT funder_id, display_name, ror_id, doi FROM openalex.common.funder WHERE funder_id = 4320328501
),
awards_transformed AS (
    SELECT
        abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(g.funder_award_id)))) % 9000000000 as id,
        g.title as display_name,
        g.description as description,
        f.funder_id,
        g.funder_award_id as funder_award_id,
        CASE WHEN TRY_CAST(g.amount AS DOUBLE) > 0 THEN TRY_CAST(g.amount AS DOUBLE) ELSE NULL END as amount,
        CASE WHEN TRY_CAST(g.amount AS DOUBLE) > 0 THEN 'EUR' ELSE NULL END as currency,
        struct(
            CONCAT('https://openalex.org/F', f.funder_id) as id,
            f.display_name, f.ror_id, f.doi
        ) as funder,
        'research' as funding_type,
        g.funding_type_raw as funder_scheme,
        'research_fi_business_finland' as provenance,
        CASE WHEN TRY_CAST(g.start_year AS INT) IS NOT NULL
             THEN CAST(CONCAT(g.start_year, '-01-01') AS DATE) ELSE NULL END as start_date,
        CASE WHEN TRY_CAST(g.end_year AS INT) IS NOT NULL
             THEN CAST(CONCAT(g.end_year, '-12-31') AS DATE) ELSE NULL END as end_date,
        TRY_CAST(g.start_year AS INT) as start_year,
        TRY_CAST(g.end_year AS INT) as end_year,
        CASE
            WHEN g.pi_family_name IS NOT NULL OR g.institution_name IS NOT NULL THEN
                struct(
                    g.pi_given_name as given_name,
                    g.pi_family_name as family_name,
                    CAST(NULL AS STRING) as orcid,
                    struct(
                        g.institution_name as name,
                        'Finland' as country,
                        CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) as ids
                    ) as affiliation,
                    CAST(NULL AS DATE) as role_start
                )
            ELSE NULL
        END as lead_investigator,
        CAST(NULL AS STRUCT<
            given_name:STRING, family_name:STRING, orcid:STRING,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>, role_start:DATE
        >) as co_lead_investigator,
        CAST(NULL AS ARRAY<STRUCT<
            given_name:STRING, family_name:STRING, orcid:STRING,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>, role_start:DATE
        >>) as investigators,
        'https://research.fi/en/results?type=funding' as landing_page_url,
        CAST(NULL AS STRING) as doi,
        concat('https://api.openalex.org/works?filter=awards.id:G', abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(g.funder_award_id)))) % 9000000000) as works_api_url,
        current_timestamp() as created_date,
        current_timestamp() as updated_date
    FROM openalex.awards.business_finland_raw g
    CROSS JOIN bf_funder f
    WHERE g.funder_award_id IS NOT NULL AND TRIM(CAST(g.funder_award_id AS STRING)) != ''
)
SELECT * FROM awards_transformed;

In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'research_fi_business_finland' AND priority = 213;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id, amount, currency,
    funder, funding_type, funder_scheme, provenance, start_date, end_date,
    start_year, end_year, lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url, created_date, updated_date,
    213 as priority
FROM openalex.awards.business_finland_awards;

## Verification Queries

In [ ]:
%sql
SELECT COUNT(*) as total_business_finland_awards FROM openalex.awards.business_finland_awards;

In [ ]:
%sql
SELECT funder_award_id, display_name, amount, currency, start_year, end_year,
       lead_investigator.given_name, lead_investigator.family_name, lead_investigator.affiliation.name
FROM openalex.awards.business_finland_awards LIMIT 10;

In [ ]:
%sql
-- §6.4a freq check. NOTE: n>5 combos are REAL prolific Finnish professors (Mikkonen/Kurvinen/Turunen/Valkama) leading many BF co-innovation projects — verified by title/institution diversity. Not a scraper bug. Do not NULL.
SELECT lead_investigator.given_name AS g, lead_investigator.family_name AS f, COUNT(*) AS n
FROM openalex.awards.business_finland_awards GROUP BY 1,2 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
SELECT
    COUNT(*) as total, COUNT(amount) as has_amount,
    COUNT(lead_investigator.family_name) as has_pi,
    COUNT(lead_investigator.affiliation.name) as has_inst, COUNT(start_date) as has_start,
    ROUND(try_divide(COUNT(amount) * 100.0, COUNT(*)), 1) as pct_amount,
    ROUND(try_divide(COUNT(lead_investigator.affiliation.name) * 100.0, COUNT(*)), 1) as pct_inst
FROM openalex.awards.business_finland_awards;

In [ ]:
%sql
SELECT start_year, COUNT(*) FROM openalex.awards.business_finland_awards
WHERE start_year IS NOT NULL GROUP BY 1 ORDER BY 1 DESC LIMIT 20;

In [ ]:
%sql
SELECT lead_investigator.affiliation.name as inst, COUNT(*) as n
FROM openalex.awards.business_finland_awards WHERE lead_investigator.affiliation.name IS NOT NULL
GROUP BY 1 ORDER BY n DESC LIMIT 20;